# 📊 Laboratorio Interactivo de Distribuciones de Probabilidad
**Cátedra:** Probabilidad y Estadística (PyE)

Este notebook está diseñado para ejecutarse en **Google Colab** y explorar de forma interactiva el comportamiento de las principales distribuciones de probabilidad discretas y continuas. 

### 🎯 Objetivos:
1. **Análisis teórico-práctico:** Comprender las funciones de probabilidad puntual (PMF), densidad (PDF) y acumulada (CDF).
2. **Sensibilidad de parámetros:** Observar cómo cambia la forma, el centro y la dispersión al modificar los parámetros clave mediante *widgets* interactivos.
3. **Simulación y convergencia:** Comprobar la **Ley de los Grandes Números** comparando muestras simuladas ($N$) contra la distribución teórica.

In [ ]:
# Configuración e importación de librerías
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual

# Estilo gráfico
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

print("✅ Entorno configurado correctamente. ¡Puedes empezar a ejecutar las celdas!")

---
# 🔵 PARTE 1: DISTRIBUCIONES DISCRETAS

Para una variable aleatoria discreta $X$:
- **Función de Probabilidad Puntual (PMF):** $p(k) = P(X = k)$
- **Función de Distribución Acumulada (CDF):** $F(k) = P(X \le k) = \sum_{x_i \le k} p(x_i)$

### 1.1 Distribución Binomial: $X \sim \text{Bin}(n, p)$
Modela el número de éxitos en $n$ ensayos independientes de Bernoulli con probabilidad constante de éxito $p$.
- **PMF:** $P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}$
- **Esperanza:** $E[X] = n \cdot p$
- **Varianza:** $Var(X) = n \cdot p \cdot (1-p)$

#### 🧠 Intuición: ¿qué significa cada parámetro?
- **$n$** = cuántos tiros hacés (la cantidad de ensayos independientes).
- **$p$** = la puntería en **cada tiro individual** (la probabilidad de acertar UNA vez, no en total).

Pensalo como un jugador de básquet tirando $n$ tiros libres, cada uno con probabilidad $p$ de entrar. Si aumentás $n$, tirás más veces (la curva se corre a la derecha y se ensancha, porque hay más aciertos posibles). Si aumentás $p$, cada tiro individual es más certero, así que esperás más aciertos sin necesidad de tirar más veces.


In [ ]:
def lab_binomial(n=10, p=0.5, N=1000, seed=42):
    np.random.seed(seed)
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    
    x = np.arange(0, n + 1)
    pmf = stats.binom.pmf(x, n, p)
    cdf = stats.binom.cdf(x, n, p)
    
    # Simulación de muestras
    muestra = stats.binom.rvs(n, p, size=N)
    
    # Gráfico 1: PMF vs Simulación
    bins = np.arange(-0.5, n + 1.5, 1)
    axs[0].hist(muestra, bins=bins, density=True, alpha=0.4, color='#3498db', edgecolor='black', label=f'Muestra (N={N})')
    axs[0].vlines(x, 0, pmf, colors='#e74c3c', linewidth=2.5, label='PMF Teórica')
    axs[0].plot(x, pmf, 'ro', ms=6)
    axs[0].set_title(f'Binomial({n}, {p}) - PMF y Simulación')
    axs[0].set_xlabel('Número de éxitos (k)')
    axs[0].set_ylabel('Probabilidad / Frecuencia')
    axs[0].legend()
    
    # Gráfico 2: CDF
    axs[1].step(x, cdf, where='post', color='#2ecc71', linewidth=2.5, label='CDF Teórica')
    axs[1].set_title(f'Binomial({n}, {p}) - Acumulada (CDF)')
    axs[1].set_xlabel('k')
    axs[1].set_ylabel('F(k) = P(X <= k)')
    axs[1].set_ylim(-0.05, 1.05)
    axs[1].legend()
    
    plt.tight_layout()
    plt.show()

interact(lab_binomial, 
         n=widgets.IntSlider(min=1, max=50, step=1, value=10, description='Ensayos (n):'),
         p=widgets.FloatSlider(min=0.01, max=0.99, step=0.05, value=0.5, description='Prob (p):'),
         N=widgets.SelectionSlider(options=[30, 100, 500, 1000, 5000, 20000], value=1000, description='Muestra (N):'),
         seed=fixed(42));

### 1.2 Distribución Poisson: $X \sim \text{Poisson}(\lambda)$
Modela el número de eventos que ocurren en un intervalo fijo de tiempo o espacio.
- **PMF:** $P(X = k) = \frac{\lambda^k e^{-\lambda}}{k!}$
- **Esperanza y Varianza:** $E[X] = Var(X) = \lambda$

#### 🧠 Intuición: ¿qué significa el parámetro?
- **$\lambda$** = la tasa promedio de ocurrencia del fenómeno en el intervalo que estás mirando (por ejemplo, clientes por hora, autos por minuto).

A diferencia de la Binomial, acá no hay un "$n$" de intentos: los eventos pueden pasar en cualquier instante, sin límite. $\lambda$ hace doble trabajo: es a la vez el centro (esperanza) y la dispersión (varianza) de la distribución. Un proceso más "movido" ($\lambda$ grande) no solo tiene más eventos en promedio, sino que también varía más de un intervalo a otro.


In [ ]:
def lab_poisson(lam=5.0, N=1000, seed=42):
    np.random.seed(seed)
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    
    # Rango visible de k según lambda
    k_max = int(stats.poisson.ppf(0.999, lam)) + 2
    x = np.arange(0, k_max)
    
    pmf = stats.poisson.pmf(x, lam)
    cdf = stats.poisson.cdf(x, lam)
    muestra = stats.poisson.rvs(lam, size=N)
    
    bins = np.arange(-0.5, k_max + 0.5, 1)
    axs[0].hist(muestra, bins=bins, density=True, alpha=0.4, color='#9b59b6', edgecolor='black', label=f'Muestra (N={N})')
    axs[0].vlines(x, 0, pmf, colors='#e67e22', linewidth=2.5, label='PMF Teórica')
    axs[0].plot(x, pmf, 'o', color='#e67e22', ms=5)
    axs[0].set_title(f'Poisson(λ={lam}) - PMF y Simulación')
    axs[0].set_xlabel('k (eventos)')
    axs[0].set_ylabel('Probabilidad')
    axs[0].legend()
    
    axs[1].step(x, cdf, where='post', color='#2ecc71', linewidth=2.5, label='CDF Teórica')
    axs[1].set_title(f'Poisson(λ={lam}) - Acumulada (CDF)')
    axs[1].set_xlabel('k')
    axs[1].set_ylabel('F(k) = P(X <= k)')
    axs[1].set_ylim(-0.05, 1.05)
    axs[1].legend()
    
    plt.tight_layout()
    plt.show()

interact(lab_poisson,
         lam=widgets.FloatSlider(min=0.5, max=30.0, step=0.5, value=5.0, description='Tasa (λ):'),
         N=widgets.SelectionSlider(options=[30, 100, 500, 1000, 5000, 20000], value=1000, description='Muestra (N):'),
         seed=fixed(42));

### 1.3 Distribución Geométrica: $X \sim \text{Geom}(p)$
Modela el número de ensayos requeridos hasta obtener el **primer éxito**.
- **PMF:** $P(X = k) = (1-p)^{k-1} p, \quad k \in \{1, 2, 3, ...\}$
- **Esperanza:** $E[X] = \frac{1}{p}$
- **Propiedad clave:** Falta de memoria.

#### 🧠 Intuición: ¿qué significa el parámetro?
- **$p$** = la probabilidad de éxito en cada intento, igual que en la Binomial — pero acá no fijamos de antemano cuántos intentos vamos a hacer: seguimos tirando hasta el primer éxito.

Pensalo como tirar un dado hasta que sale el primer 6 ($p=1/6$). Si $p$ es grande (tirador muy certero), el primer éxito llega casi enseguida, y la distribución se concentra en valores chicos de $k$. Si $p$ es chico, puede haber que esperar muchos intentos, y aparece una cola larga hacia la derecha.


In [ ]:
def lab_geometrica(p=0.3, N=1000, seed=42):
    np.random.seed(seed)
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    
    k_max = int(stats.geom.ppf(0.999, p)) + 1
    x = np.arange(1, k_max + 1)
    
    pmf = stats.geom.pmf(x, p)
    cdf = stats.geom.cdf(x, p)
    muestra = stats.geom.rvs(p, size=N)
    
    bins = np.arange(0.5, k_max + 1.5, 1)
    axs[0].hist(muestra, bins=bins, density=True, alpha=0.4, color='#1abc9c', edgecolor='black', label=f'Muestra (N={N})')
    axs[0].vlines(x, 0, pmf, colors='#c0392b', linewidth=2.5, label='PMF Teórica')
    axs[0].plot(x, pmf, 'ro', ms=5)
    axs[0].set_title(f'Geométrica(p={p}) - PMF y Simulación')
    axs[0].set_xlabel('k (ensayo del 1er éxito)')
    axs[0].set_ylabel('Probabilidad')
    axs[0].legend()
    
    axs[1].step(x, cdf, where='post', color='#2ecc71', linewidth=2.5, label='CDF Teórica')
    axs[1].set_title(f'Geométrica(p={p}) - Acumulada (CDF)')
    axs[1].set_xlabel('k')
    axs[1].set_ylabel('F(k) = P(X <= k)')
    axs[1].set_ylim(-0.05, 1.05)
    axs[1].legend()
    
    plt.tight_layout()
    plt.show()

interact(lab_geometrica,
         p=widgets.FloatSlider(min=0.05, max=0.95, step=0.05, value=0.3, description='Prob (p):'),
         N=widgets.SelectionSlider(options=[30, 100, 500, 1000, 5000, 20000], value=1000, description='Muestra (N):'),
         seed=fixed(42));

---
# 🔴 PARTE 2: DISTRIBUCIONES CONTINUAS

Para una variable aleatoria continua $X$:
- **Función de Densidad de Probabilidad (PDF):** $f(x) \ge 0$, con $\int_{-\infty}^{\infty} f(x)dx = 1$
- **Probabilidad de Intervalos:** $P(a \le X \le b) = \int_a^b f(x)dx = F(b) - F(a)$
- **Función Acumulada (CDF):** $F(x) = P(X \le x) = \int_{-\infty}^x f(t)dt$

### 2.1 Distribución Normal (Gaussiana): $X \sim \mathcal{N}(\mu, \sigma^2)$
La distribución continua más relevante en estadística (Teorema Central del Límite).
- **PDF:** $f(x) = \frac{1}{\sigma \sqrt{2\pi}} e^{-\frac{1}{2}\left(\frac{x-\mu}{\sigma}\right)^2}$
- **Parámetros:** $\mu$ (media / centro de simetría), $\sigma$ (desviación estándar / dispersión).

#### 🧠 Intuición: ¿qué significa cada parámetro?
- **$\mu$** = hacia dónde apunta el tirador (el centro, el valor que en promedio da en el blanco).
- **$\sigma$** = cuánto tiende a errar: qué tan dispersos quedan los tiros alrededor del centro.

Un tirador con $\sigma$ chico agrupa los tiros muy cerca de $\mu$ (campana angosta y alta); uno con $\sigma$ grande los desparrama (campana achatada y ancha). Fijate en el gráfico: mover $\mu$ solo **traslada** la campana sin cambiar su forma, mientras que mover $\sigma$ cambia el **ancho** pero no el centro.


In [ ]:
def lab_normal(mu=0.0, sigma=1.0, N=1000, seed=42):
    np.random.seed(seed)
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    
    # Rango de graficación [-4 sigma, +4 sigma]
    x = np.linspace(mu - 4*sigma - 2, mu + 4*sigma + 2, 500)
    pdf = stats.norm.pdf(x, loc=mu, scale=sigma)
    cdf = stats.norm.cdf(x, loc=mu, scale=sigma)
    muestra = stats.norm.rvs(loc=mu, scale=sigma, size=N)
    
    # Gráfico 1: Histograma vs PDF
    axs[0].hist(muestra, bins=35, density=True, alpha=0.4, color='#e74c3c', edgecolor='black', label=f'Muestra (N={N})')
    axs[0].plot(x, pdf, color='#2c3e50', linewidth=2.5, label='PDF Teórica f(x)')
    axs[0].set_title(f'Normal(μ={mu}, σ={sigma}) - Densidad (PDF)')
    axs[0].set_xlabel('x')
    axs[0].set_ylabel('Densidad')
    axs[0].legend()
    
    # Gráfico 2: CDF
    axs[1].plot(x, cdf, color='#2ecc71', linewidth=2.5, label='CDF Teórica F(x)')
    axs[1].set_title(f'Normal(μ={mu}, σ={sigma}) - Acumulada (CDF)')
    axs[1].set_xlabel('x')
    axs[1].set_ylabel('F(x) = P(X <= x)')
    axs[1].set_ylim(-0.05, 1.05)
    axs[1].legend()
    
    plt.tight_layout()
    plt.show()

interact(lab_normal,
         mu=widgets.FloatSlider(min=-10.0, max=10.0, step=0.5, value=0.0, description='Media (μ):'),
         sigma=widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description='Desv. (σ):'),
         N=widgets.SelectionSlider(options=[30, 100, 500, 1000, 5000, 20000], value=1000, description='Muestra (N):'),
         seed=fixed(42));

### 2.2 Distribución Exponencial: $X \sim \text{Exp}(\lambda)$
Modela el tiempo transcurrido entre eventos de un proceso de Poisson.
- **PDF:** $f(x) = \lambda e^{-\lambda x}, \quad x \ge 0$
- **Esperanza:** $E[X] = \frac{1}{\lambda}$
- En `scipy.stats.expon`, el parámetro `scale` equivale a $\frac{1}{\lambda}$.

#### 🧠 Intuición: ¿qué significa el parámetro?
- **$\lambda$** = de nuevo una tasa, la misma idea que en Poisson: cuántos eventos ocurren en promedio por unidad de tiempo.

Pero acá no contamos eventos, medimos el **tiempo** hasta el próximo. Pensalo como la espera entre colectivos: si $\lambda$ es grande (pasan muchos colectivos por hora), la espera tiende a ser corta (densidad concentrada cerca de 0). Si $\lambda$ es chica (pasan pocos), hay que esperar más, y la densidad se estira hacia la derecha.


In [ ]:
def lab_exponencial(lam=1.0, N=1000, seed=42):
    np.random.seed(seed)
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    
    scale = 1.0 / lam
    x = np.linspace(0, stats.expon.ppf(0.999, scale=scale), 500)
    
    pdf = stats.expon.pdf(x, scale=scale)
    cdf = stats.expon.cdf(x, scale=scale)
    muestra = stats.expon.rvs(scale=scale, size=N)
    
    axs[0].hist(muestra, bins=35, density=True, alpha=0.4, color='#f39c12', edgecolor='black', label=f'Muestra (N={N})')
    axs[0].plot(x, pdf, color='#2c3e50', linewidth=2.5, label='PDF Teórica')
    axs[0].set_title(f'Exponencial(λ={lam}) - PDF y Simulación')
    axs[0].set_xlabel('x (tiempo/espacio)')
    axs[0].set_ylabel('Densidad')
    axs[0].legend()
    
    axs[1].plot(x, cdf, color='#2ecc71', linewidth=2.5, label='CDF Teórica')
    axs[1].set_title(f'Exponencial(λ={lam}) - Acumulada (CDF)')
    axs[1].set_xlabel('x')
    axs[1].set_ylabel('F(x) = P(X <= x)')
    axs[1].set_ylim(-0.05, 1.05)
    axs[1].legend()
    
    plt.tight_layout()
    plt.show()

interact(lab_exponencial,
         lam=widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description='Tasa (λ):'),
         N=widgets.SelectionSlider(options=[30, 100, 500, 1000, 5000, 20000], value=1000, description='Muestra (N):'),
         seed=fixed(42));

### 2.3 Distribución Uniforme Continua: $X \sim U(a, b)$
Asigna densidad constante a todos los puntos dentro del intervalo $[a, b]$.
- **PDF:** $f(x) = \frac{1}{b - a}, \quad a \le x \le b$
- **Esperanza:** $E[X] = \frac{a+b}{2}$

#### 🧠 Intuición: ¿qué significa cada parámetro?
- **$a$, $b$** = los extremos del intervalo donde puede caer $X$.

La clave de la Uniforme es que **no hay ningún valor preferido** dentro de $[a,b]$: es el modelo de "no sé nada más que el rango posible". Ensanchar el intervalo reparte la misma probabilidad total (área $1$) en un rango más grande, así que la densidad (la altura del rectángulo) baja — hay más lugares posibles, entonces cada uno pesa menos por unidad de longitud.


In [ ]:
def lab_uniforme(a=0.0, b=10.0, N=1000, seed=42):
    if a >= b:
        print("⚠️ Error: El valor de 'a' debe ser menor que 'b'.")
        return
        
    np.random.seed(seed)
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    
    loc = a
    scale = b - a
    x = np.linspace(a - 0.2*scale, b + 0.2*scale, 500)
    
    pdf = stats.uniform.pdf(x, loc=loc, scale=scale)
    cdf = stats.uniform.cdf(x, loc=loc, scale=scale)
    muestra = stats.uniform.rvs(loc=loc, scale=scale, size=N)
    
    axs[0].hist(muestra, bins=25, density=True, alpha=0.4, color='#34495e', edgecolor='black', label=f'Muestra (N={N})')
    axs[0].plot(x, pdf, color='#e74c3c', linewidth=2.5, label='PDF Teórica')
    axs[0].set_title(f'Uniforme Continuous U({a}, {b})')
    axs[0].set_xlabel('x')
    axs[0].set_ylabel('Densidad')
    axs[0].legend()
    
    axs[1].plot(x, cdf, color='#2ecc71', linewidth=2.5, label='CDF Teórica')
    axs[1].set_title(f'Uniforme Continuous U({a}, {b}) - CDF')
    axs[1].set_xlabel('x')
    axs[1].set_ylabel('F(x) = P(X <= x)')
    axs[1].set_ylim(-0.05, 1.05)
    axs[1].legend()
    
    plt.tight_layout()
    plt.show()

interact(lab_uniforme,
         a=widgets.FloatSlider(min=-10.0, max=10.0, step=0.5, value=0.0, description='Límite inf (a):'),
         b=widgets.FloatSlider(min=-5.0, max=20.0, step=0.5, value=10.0, description='Límite sup (b):'),
         N=widgets.SelectionSlider(options=[30, 100, 500, 1000, 5000, 20000], value=1000, description='Muestra (N):'),
         seed=fixed(42));

---
## 💡 Preguntas de Reflexión para el Alumno:
1. **Efecto de la Muestra ($N$):** ¿Qué ocurre con la forma del histograma cuando aumentas $N$ de $30$ a $20,000$? ¿Cómo se relaciona esto con la definición de probabilidad frecuentista?
2. **Simetría y Sesgo:** En la **Binomial**, ¿para qué valores de $p$ la distribución es completamente simétrica? ¿Qué sucede cuando $p$ se acerca a $0$ o a $1$?
3. **Poisson vs Binomial:** Si en la Binomial mantienes $E[X] = n \cdot p = \lambda$ constante (por ejemplo $n=100, p=0.05 \Rightarrow \lambda=5$), ¿qué observas al comparar su forma con la distribución de Poisson?

---
## ✏️ Ejercicios

**Ejercicio 1 (Normal).** Un tirador apunta al $10$ ($\mu=10$) y querés que aproximadamente el $95\%$ de sus tiros caiga entre $8$ y $12$. Usando la regla empírica ($\mu\pm2\sigma\approx95\%$), estimá a mano qué valor de $\sigma$ deberías usar. Después ajustá el slider en el laboratorio de la Normal con ese $\sigma$ y generá una muestra grande ($N=20000$): ¿qué fracción de los datos simulados cae realmente en $[8,12]$?

**Ejercicio 2 (Exponencial y Ley de los Grandes Números).** En el laboratorio de la Exponencial, fijá $\lambda=0.5$ (tiempo medio de espera $=2$). Sin cambiar $\lambda$, aumentá $N$ desde $30$ hasta $20000$ y compará el histograma simulado contra la curva teórica en cada paso. ¿Qué ley te garantiza que, a medida que $N$ crece, el histograma se parezca cada vez más a la densidad teórica?
